In [7]:
import mediapipe as mp
import cv2
import numpy as np
import time
import threading
from collections import deque

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path = "face_landmarker.task"

latest_result = None
result_lock = threading.Lock()

def store_result(result, output_image, timestamp_ms):
    global latest_result
    with result_lock:
        latest_result = result

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=store_result
)



In [ ]:
from scipy.signal import butter, filtfilt

def chrom(rgb_signal):
    r = rgb_signal[:, 0].astype(np.float64)
    g = rgb_signal[:, 1].astype(np.float64)
    b = rgb_signal[:, 2].astype(np.float64)

    r_mean = np.mean(r) + 1e-6
    g_mean = np.mean(g) + 1e-6
    b_mean = np.mean(b) + 1e-6

    r_n = r / r_mean
    g_n = g / g_mean
    b_n = b / b_mean

    Xs = 3 * r_n - 2 * g_n
    Ys = 1.5 * r_n + g_n - 1.5 * b_n

    std_Xs = np.std(Xs) + 1e-6
    std_Ys = np.std(Ys) + 1e-6
    alpha = std_Xs / std_Ys

    pulse = Xs - alpha * Ys
    return pulse

def bandpass(signal, fps, low=0.7, high=3.5):
    nyq = fps / 2.0
    low_n = low / nyq
    high_n = high / nyq
    b, a = butter(4, [low_n, high_n], btype='band')
    return filtfilt(b, a, signal)

def estimate_hr(pulse_signal, fps):
    n = len(pulse_signal)

    n_padded = n * 4
    freqs = np.fft.rfftfreq(n_padded, d=1.0 / fps)
    fft_mag = np.abs(np.fft.rfft(pulse_signal, n=n_padded))

    valid = (freqs >= 0.7) & (freqs <= 3.5)
    if not np.any(valid):
        return 0

    peak_freq = freqs[valid][np.argmax(fft_mag[valid])]
    return peak_freq * 60

In [ ]:
regions = {
    "forehead": [109, 10, 338, 336, 9, 107],
    "left_cheek": [116, 111, 117, 118, 119, 120, 100, 142, 36, 205, 123],
    "right_cheek": [371, 329, 349, 348, 347, 346, 340, 345, 352, 425, 266]
}

cap = cv2.VideoCapture(0)
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps == 0:
    fps = 30
fps = float(fps)

window_seconds = 7
buffer_size = int(fps * window_seconds)

rgb_buffer = deque(maxlen=buffer_size)

bpm_display = 0
last_hr_time = time.time()
hr_update_interval = 1.5  

start_time = time.time()
time.sleep(1)

with FaceLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int((time.time() - start_time) * 1000)
        landmarker.detect_async(mp_image, timestamp_ms)

        with result_lock:
            current_result = latest_result

        if current_result and current_result.face_landmarks:
            landmarks = current_result.face_landmarks[0]
            h, w, _ = frame.shape

            for lm in landmarks:
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

            mask = np.zeros(frame.shape[:2], dtype=np.uint8)
            for region_name, ids in regions.items():
                points = np.array([
                    (int(landmarks[i].x * w), int(landmarks[i].y * h))
                    for i in ids
                ])
                cv2.polylines(frame, [points], isClosed=True, color=(0, 0, 255), thickness=2)
                cv2.fillPoly(mask, [points], 255)

            mean_bgr = cv2.mean(frame, mask=mask)[:3]
            B, G, R = mean_bgr
            rgb_buffer.append([R, G, B])

            now = time.time()
            if len(rgb_buffer) >= buffer_size and (now - last_hr_time) >= hr_update_interval:
                last_hr_time = now

                signal = np.array(rgb_buffer)   
                pulse = chrom(signal)            # colour channel averaging
                filtered = bandpass(pulse, fps)  # bandpass
                bpm_display = estimate_hr(filtered, fps)  # peak detection (fourier transform

        #display heartrate
        color = (0, 255, 255) if bpm_display > 0 else (100, 100, 100)
        cv2.putText(frame, f"HR: {bpm_display:.1f} BPM",
                    (30, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1.2, color, 2)
        cv2.putText(frame, f"Buffer: {len(rgb_buffer)}/{buffer_size}",
                    (30, 90), cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (200, 200, 200), 1)

        cv2.imshow("rPPG", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

W0000 00:00:1781251969.256251 17224628 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781251969.259284 17224628 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1781251969.260403 17224631 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781251969.268234 17224631 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
E0000 00:00:1781252090.284435 17224629 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-06-12T16:28:50.279604+08:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180
E0000 00:00:1781252146.294962 17224629 portable_clearcut_uploader.cc:90] Failed to send to cle

KeyboardInterrupt: 